In [ ]:
"""
This module generates Critical Difference (CD) diagrams for statistical comparison
of outlier detection algorithms across multiple metrics and conversion methods.

CD diagrams provide a visual representation of algorithm performance rankings and
statistically significant differences based on the Nemenyi post-hoc test.

Key Features:
  - Load performance metrics from CSV files
  - Generate CD diagrams for multiple metrics
  - Support for binary and non-binary conversion methods
  - Batch processing with labeled output
  - Statistical significance visualization

Organization:
  1. Installation & Dependencies
  2. Configuration & Constants
  3. Data Loading Functions
  4. CD Diagram Generation Functions
  5. Main Execution Pipeline
"""

import os
import pandas as pd
import re
import csv
import json
from typing import Dict, List, Tuple
import warnings

from CriticalDifference import draw_cd_diagram

# Suppress all library warnings for cleaner output
warnings.filterwarnings("ignore")

In [ ]:
# =============================================================================
# SECTION 1: Configuration & Constants
# =============================================================================

# Conversion methods configuration
CONVERSION_METHODS = {
    'non_binary': {
        'input_file': r'..\results\conversion_methods\critical_non-binary.csv',
        'output_prefix': r'..\results\conversion_methods\non_binary-',
        'description': 'Non-binary conversion methods (EXC, EXCDOWN, GRO, GRODOWN)'
    },
    'binary': {
        'input_file': r'..\results\conversion_methods\critical_binary.csv',
        'output_prefix': r'..\results\conversion_methods\binary-',
        'description': 'Binary conversion methods (BIN, BINDOWN)'
    }
}

'''
dataset_name 	classifier_name 	accuracy 	metric
0 	cardiotocography.csv 	GRO 	0.xxx 	AUC
1 	cardiotocography.csv 	GRO 	0.xxx 	P@n
2 	cardiotocography.csv 	GRO 	0.xxx 	AP
3 	cardiotocography.csv 	GRO 	0.xxx 	Max-F1
4 	digits.csv 	            GRO 	0.xxx 	AUC
... 	... 	... 	... 	...
204 	wine.csv 	        EXCDOWN 	0.xxx 	AUC
205 	wine.csv 	        EXCDOWN 	0.xxx 	P@n
206 	wine.csv 	        EXCDOWN 	0.xxx 	AP
207 	wine.csv 	        EXCDOWN 	0.xxx 	Max-F1
'''

# CD Diagram parameters
CD_DIAGRAM_DEFAULTS = {
    'labels': True,
    'figsize': (10, 6),
    'font_size': 10
}

# File encoding
FILE_ENCODING = 'utf-8'
CSV_SEPARATOR = ';'


# =============================================================================
# SECTION 2: Data Loading Functions
# =============================================================================

def load_performance_data(file_path: str, separator: str = CSV_SEPARATOR) -> pd.DataFrame:
    """
    Load performance metrics from CSV file.
    
    Reads performance data containing algorithm rankings and scores for
    different metrics across datasets.
    
    Args:
        file_path: Path to CSV file containing performance metrics
        separator: CSV delimiter (default: ';')
        
    Returns:
        DataFrame with columns including 'metric' and algorithm performance columns
        
    Raises:
        FileNotFoundError: If specified file does not exist
        pd.errors.ParserError: If CSV parsing fails
    """
    # Check if file exists
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Performance data file not found: {file_path}")
    
    try:
        # Load CSV file with specified separator
        df = pd.read_csv(file_path, sep=separator)
        print(f"✓ Loaded performance data from: {file_path}")
        print(f"  Shape: {df.shape}")
        print(f"  Columns: {list(df.columns)}")
        
        return df
    
    except pd.errors.ParserError as e:
        raise pd.errors.ParserError(f"Error parsing CSV file {file_path}: {e}")
    except Exception as e:
        raise Exception(f"Unexpected error loading {file_path}: {e}")


def validate_performance_data(df: pd.DataFrame) -> bool:
    """
    Validate that performance data has required structure.
    
    Checks for presence of 'metric' column and sufficient data.
    
    Args:
        df: DataFrame to validate
        
    Returns:
        True if valid, raises exception otherwise
    """
    # Check for required 'metric' column
    if 'metric' not in df.columns:
        raise ValueError("Performance data must contain 'metric' column")
    
    # Check for sufficient data
    if len(df) == 0:
        raise ValueError("Performance data is empty")
    
    # Check for algorithm performance columns (at least one besides 'metric')
    if len(df.columns) < 2:
        raise ValueError("Performance data must contain algorithm columns")
    
    return True


def get_unique_metrics(df: pd.DataFrame) -> List[str]:
    """
    Extract unique metrics from performance data.
    
    Args:
        df: Performance DataFrame with 'metric' column
        
    Returns:
        Sorted list of unique metric names
    """
    metrics = df['metric'].unique().tolist()
    return sorted(metrics)


def get_metric_data(df: pd.DataFrame, metric: str) -> pd.DataFrame:
    """
    Extract data for a specific metric, removing the metric column.
    
    Args:
        df: Performance DataFrame
        metric: Metric name to filter
        
    Returns:
        DataFrame containing only data for specified metric, without 'metric' column
    """
    # Filter for specified metric
    df_filtered = df.query('metric == @metric')
    
    # Remove metric column for CD diagram input
    df_filtered = df_filtered.drop(columns=['metric'])
    
    return df_filtered


# =============================================================================
# SECTION 3: CD Diagram Generation Functions
# =============================================================================

def draw_single_cd_diagram(df_perf: pd.DataFrame, metric: str, 
                          output_path: str, use_labels: bool = True) -> bool:
    """
    Generate and save a single Critical Difference diagram.
    
    Creates a CD diagram for the given performance data and metric,
    visualizing algorithm rankings and statistically significant differences.
    
    Args:
        df_perf: Performance data DataFrame for specific metric
        metric: Metric name (for titling)
        output_path: Path to save output image (without extension)
        use_labels: Whether to display algorithm labels on diagram
        
    Returns:
        True if successful, False otherwise
    """
    try:
        print(f"\n  Generating CD diagram for metric: {metric}")
        
        # Draw Critical Difference diagram
        # Note: draw_cd_diagram saves to output_path.pdf
        draw_cd_diagram(
            df_perf=df_perf,
            title=f'Critical Difference Diagram - {metric}',
            labels=use_labels,
            output=output_path
        )
        
        print(f"  ✓ Saved to: {output_path}.pdf")
        return True
    
    except Exception as e:
        print(f"  ✗ Error generating CD diagram for {metric}: {e}")
        return False


def generate_cd_diagrams_for_method(input_file: str, output_prefix: str,
                                   method_name: str) -> Tuple[int, int]:
    """
    Generate CD diagrams for all metrics from a single conversion method.
    
    Main processing function that:
    1. Loads performance data
    2. Validates data structure
    3. Extracts unique metrics
    4. Generates CD diagram for each metric
    5. Saves with appropriate naming
    
    Args:
        input_file: Path to input CSV file
        output_prefix: Output path prefix for saving diagrams
        method_name: Name of conversion method (for logging)
        
    Returns:
        Tuple of (successful_diagrams, failed_diagrams)
    """
    print(f"\n{'='*70}")
    print(f"Processing: {method_name}")
    print(f"{'='*70}")
    
    # ====================================================================
    # Step 1: Load and validate performance data
    # ====================================================================
    try:
        df = load_performance_data(input_file)
        validate_performance_data(df)
    except (FileNotFoundError, ValueError, Exception) as e:
        print(f"✗ Failed to load data: {e}")
        return 0, 0
    
    # ====================================================================
    # Step 2: Extract unique metrics
    # ====================================================================
    metrics = get_unique_metrics(df)
    print(f"\n✓ Found {len(metrics)} unique metrics:")
    for i, metric in enumerate(metrics, 1):
        print(f"  {i}. {metric}")
    
    # ====================================================================
    # Step 3: Generate CD diagram for each metric
    # ====================================================================
    successful = 0
    failed = 0
    
    print(f"\nGenerating CD diagrams...")
    for metric in metrics:
        # Extract data for this metric
        df_metric = get_metric_data(df, metric)
        
        # Generate output path
        output_path = f"{output_prefix}{metric}"
        
        # Draw CD diagram
        if draw_single_cd_diagram(df_metric, metric, output_path):
            successful += 1
        else:
            failed += 1
    
    # ====================================================================
    # Step 4: Print summary
    # ====================================================================
    print(f"\n{'='*70}")
    print(f"Summary for {method_name}")
    print(f"{'='*70}")
    print(f"✓ Successfully generated: {successful} diagram(s)")
    if failed > 0:
        print(f"✗ Failed to generate: {failed} diagram(s)")
    print()
    
    return successful, failed



In [ ]:
# =============================================================================
# SECTION 4: Main Execution Pipeline
# =============================================================================

def main():
    """
    Main execution pipeline for Critical Difference diagram generation.
    
    Orchestrates complete workflow:
    1. Configure input/output paths for all conversion methods
    2. Process each conversion method
    3. Generate CD diagrams for all metrics
    4. Report overall results
    
    Generates CD diagrams for:
    - Binary methods: BIN, BINDOWN
    - Non-binary methods: EXC, EXCDOWN, GRO, GRODOWN
    """
    print("\n" + "="*70)
    print("Critical Difference Diagram Generation Pipeline")
    print("="*70)
    
    # ====================================================================
    # Initialize tracking variables
    # ====================================================================
    total_successful = 0
    total_failed = 0
    processing_results = {}
    
    # ====================================================================
    # Process each conversion method
    # ====================================================================
    for method_key, config in CONVERSION_METHODS.items():
        print(f"\n[Processing] {method_key.upper()}")
        print(f"Description: {config['description']}")
        print(f"Input: {config['input_file']}")
        
        # Generate CD diagrams for this method
        successful, failed = generate_cd_diagrams_for_method(
            input_file=config['input_file'],
            output_prefix=config['output_prefix'],
            method_name=method_key
        )
        
        # Track results
        processing_results[method_key] = {
            'successful': successful,
            'failed': failed
        }
        total_successful += successful
        total_failed += failed
    
    # ====================================================================
    # Final Summary
    # ====================================================================
    print("\n" + "="*70)
    print("Pipeline Complete - Final Summary")
    print("="*70)
    
    # Print results per method
    for method_key, results in processing_results.items():
        print(f"\n{method_key.upper()}:")
        print(f"  ✓ Successful: {results['successful']} diagram(s)")
        print(f"  ✗ Failed: {results['failed']} diagram(s)")
    
    # Print overall results
    print(f"\n{'─'*70}")
    print(f"OVERALL RESULTS:")
    print(f"  Total Successful: {total_successful}")
    print(f"  Total Failed: {total_failed}")
    
    if total_failed == 0:
        print(f"\n✓ All diagrams generated successfully!")
    else:
        print(f"\n⚠ {total_failed} diagram(s) failed to generate. Check errors above.")
    
    print("="*70 + "\n")


if __name__ == "__main__":
    main()